# Landauer transport in a finite tight-binding chain

This notebook follows the Stage 1 calculation for a coherent finite 1D device with semi-infinite 1D leads. It uses the implementation in `src/quantum_transport`; it does not duplicate the model routines or add a broadening parameter.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import numpy as np
from quantum_transport import (
    advanced_device_green_function,
    landauer_transmission,
    lead_self_energy,
    left_broadening_matrix,
    retarded_device_green_function,
    right_broadening_matrix,
    scalar_broadening,
    surface_green_function,
    tight_binding_hamiltonian,
)

N = 10
t = 1.0
epsilon_0 = 0.0
t_c = 1.0
E = 0.5

## 1. Finite device Hamiltonian

The device has open boundaries and uses `H[i,i] = epsilon_i` and `H[i,i+1] = H[i+1,i] = -t`. There is no coupling between the first and last sites.

In [ ]:
H_clean = tight_binding_hamiltonian(N, onsite_energy=epsilon_0, hopping=t)
H_clean.shape, H_clean[:4, :4]

## 2. Lead dispersion

For the semi-infinite nearest-neighbour lead, `E(k) = epsilon_0 - 2 t cos(k)`. Its propagating band is `[epsilon_0 - 2t, epsilon_0 + 2t]`.

In [ ]:
lead_band = (epsilon_0 - 2 * t, epsilon_0 + 2 * t)
lead_band

## 3. Surface Green function

The retarded branch has negative imaginary part inside the band and is real outside. The square-root behavior needs care at the band edges; this notebook evaluates representative interior and exterior energies.

In [ ]:
g_inside = surface_green_function(E, onsite_energy=epsilon_0, hopping=t)
g_outside = surface_green_function(2.5, onsite_energy=epsilon_0, hopping=t)
g_inside, g_outside

## 4. Lead self-energy

The endpoint self-energy is `Sigma^r = t_c^2 g_s^r`; the left and right leads are identical in this benchmark.

In [ ]:
sigma = lead_self_energy(E, onsite_energy=epsilon_0, hopping=t, coupling=t_c)
sigma

## 5. Device Green function

`G^r = [E I - H_device - Sigma_L - Sigma_R]^-1`, with left and right self-energies acting on the first and last sites. The implementation solves the linear system against the identity. The advanced Green function is `(G^r)^dagger`.

In [ ]:
G_r = retarded_device_green_function(E, H_clean, sigma, sigma)
G_a = advanced_device_green_function(E, H_clean, sigma, sigma)
G_r.shape, G_r[0, 0], G_r[0, -1], np.allclose(G_a, G_r.conj().T)

## 6. Lead broadening

`Gamma = i(Sigma^r - Sigma^{r dagger})`, or `-2 Im(Sigma^r)` for a scalar self-energy. With the retarded branch, the in-band broadening is non-negative.

In [ ]:
gamma = scalar_broadening(sigma)
Gamma_L = left_broadening_matrix(N, sigma)
Gamma_R = right_broadening_matrix(N, sigma)
gamma, Gamma_L.diagonal(), Gamma_R.diagonal()

## 7. Landauer transmission

The transmission is `T(E) = Tr[Gamma_L G^r Gamma_R G^a]`. The scalar function checks that residual imaginary or negative values are no larger than its stated floating-point tolerance.

In [ ]:
T_clean = landauer_transmission(G_r, Gamma_L, Gamma_R)
T_clean

## 8. Clean matched device

For a clean device identical to the leads, the analytical expectation is unit transmission inside the propagating band away from the edges. The following values are direct calculations, with no normalization to one.

In [ ]:
sample_energies = np.array([-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5])
clean_transmissions = []
for energy in sample_energies:
    sigma_e = lead_self_energy(energy, epsilon_0, t, t_c)
    G_e = retarded_device_green_function(energy, H_clean, sigma_e, sigma_e)
    GL_e = left_broadening_matrix(N, sigma_e)
    GR_e = right_broadening_matrix(N, sigma_e)
    clean_transmissions.append(landauer_transmission(G_e, GL_e, GR_e))
list(zip(sample_energies, clean_transmissions))

## 9. Single onsite barrier

A barrier adds `U = 1.5` to zero-based site 4 (the fifth site). The lead parameters and the transport calculation remain unchanged.

In [ ]:
U = 1.5
barrier_onsite = np.full(N, epsilon_0)
barrier_onsite[4] += U
H_barrier = tight_binding_hamiltonian(N, onsite_energy=barrier_onsite, hopping=t)
barrier_transmissions = []
for energy in sample_energies:
    sigma_e = lead_self_energy(energy, epsilon_0, t, t_c)
    G_e = retarded_device_green_function(energy, H_barrier, sigma_e, sigma_e)
    GL_e = left_broadening_matrix(N, sigma_e)
    GR_e = right_broadening_matrix(N, sigma_e)
    barrier_transmissions.append(landauer_transmission(G_e, GL_e, GR_e))
list(zip(sample_energies, clean_transmissions, barrier_transmissions))

These are finite-device coherent transport results. They do not establish thermodynamic behavior and do not include disorder, interactions, inelastic scattering, or higher-dimensional transport.